
# Production one-arm WDM profile: parallel DWT and parallel links

This notebook profiles the selected full-year CPU route:

- two concurrent plus/cross WDM transforms;
- the unchanged materialized analytic-parity shift kernel;
- two concurrent shift groups containing three directed links each;
- `row_chunk_size = 32640`;
- `lag_block_size = 17`;
- complex64 shift assembly.

The notebook compares every optimized route against the ordinary production
route and requires exact numerical agreement.


In [ ]:

# --- 1. CPU-first imports and implementation fingerprint ---

import gc
import inspect
import json
import os
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

# Set before importing JAX, FEW, WDM, or WDM_LISAresponse.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    ONE_ARM_EXECUTION_MODES,
)

import WDM
import WDM_LISAresponse
import WDM.code.time_delay_filters.plans as shift_plans_module
import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM_LISAresponse.code.lisa_response.response_with_plan as response_module


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("plans.py:", shift_plans_module.__file__)
print("one_arm_plan.py:", one_arm_plan_module.__file__)
print("response_with_plan.py:", response_module.__file__)
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu"
assert all(device.platform == "cpu" for device in jax.devices())
assert "materialized_parallel_links" in ONE_ARM_EXECUTION_MODES
assert hasattr(
    shift_plans_module.VariableShiftBatchPlan,
    "apply_device_parallel_groups",
)
assert hasattr(
    OneArmWDMPlan,
    "apply_batch_device_materialized_parallel_groups",
)
assert "parallel_wdm_transforms" in inspect.signature(
    _build_wdm_dT_context_from_wdm
).parameters
assert "parallel_links_per_group" in inspect.signature(
    OneArmWDMPlan.apply_device
).parameters


In [ ]:

# --- 2. Full-year production configuration ---

N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

ROW_CHUNK_SIZE = 32640
LAG_BLOCK_SIZE = 17
ASSEMBLY_PRECISION = "complex64"
ASSEMBLY_BACKEND = "lagfirst_chunked_lagblock"

# Selected conservative link schedule.
PARALLEL_LINKS_PER_GROUP = 3
PARALLEL_MAX_WORKERS = 2

alpha = 0.005
tukey_factor = 4

ORBIT_FILE = "esa-trailing-orbits.h5"
theta = np.pi / 2
phi = 0.0

m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

DWT_WARMUPS = 2
DWT_REPEATS = 15
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 15
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 11

SAVE_RESULTS = True
RESULT_PATH = Path(
    ".benchmark_cache/one_arm_parallel_production_profile.json"
)
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

SHIFT_KWARGS = {
    "assembly_backend": ASSEMBLY_BACKEND,
    "assembly_precision": ASSEMBLY_PRECISION,
    "row_chunk_size": ROW_CHUNK_SIZE,
    "lag_block_size": LAG_BLOCK_SIZE,
    "job_block_size": 1,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
}

print("Expected WDM Nt:", N // (2 * Nf))
print("Shift configuration:", SHIFT_KWARGS)
print(
    "Parallel links:",
    f"{6 // PARALLEL_LINKS_PER_GROUP} groups × "
    f"{PARALLEL_LINKS_PER_GROUP} links",
)


In [ ]:

# --- 3. Timing, synchronization, and validation helpers ---

def block_result(value):
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def process_cpu_seconds():
    if _process is None:
        return np.nan
    values = _process.cpu_times()
    return float(values.user + values.system)


def timed_call(function):
    cpu_start = process_cpu_seconds()
    wall_start = time.perf_counter()

    result = function()
    block_result(result)

    wall_s = time.perf_counter() - wall_start
    cpu_end = process_cpu_seconds()

    cpu_s = (
        float(cpu_end - cpu_start)
        if np.isfinite(cpu_start) and np.isfinite(cpu_end)
        else np.nan
    )
    effective_cores = (
        cpu_s / wall_s
        if wall_s > 0.0 and np.isfinite(cpu_s)
        else np.nan
    )
    return result, float(wall_s), float(cpu_s), float(effective_cores)


def summarize(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "mean_s": float(np.mean(values)),
        "std_s": float(np.std(values)),
        "all_s": [float(value) for value in values],
    }


def compare_arrays(candidate, reference):
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)
    difference = candidate - reference

    max_abs = jnp.max(jnp.abs(difference))
    reference_norm = jnp.linalg.norm(reference)
    rel_l2 = jnp.where(
        reference_norm > 0,
        jnp.linalg.norm(difference) / reference_norm,
        0.0,
    )
    block_result((max_abs, rel_l2))

    return {
        "max_abs": float(max_abs),
        "rel_l2": float(rel_l2),
    }


def paired_benchmark(
    baseline_function,
    candidate_function,
    *,
    warmups,
    repeats,
):
    for _ in range(int(warmups)):
        block_result(baseline_function())
        block_result(candidate_function())

    records = {
        "baseline_wall": [],
        "candidate_wall": [],
        "baseline_cpu": [],
        "candidate_cpu": [],
        "baseline_effective": [],
        "candidate_effective": [],
    }

    baseline_output = None
    candidate_output = None

    for repeat_index in range(int(repeats)):
        order = (
            (("baseline", baseline_function), ("candidate", candidate_function))
            if repeat_index % 2 == 0
            else (("candidate", candidate_function), ("baseline", baseline_function))
        )

        for label, function in order:
            output, wall_s, cpu_s, effective = timed_call(function)

            records[f"{label}_wall"].append(wall_s)
            records[f"{label}_cpu"].append(cpu_s)
            records[f"{label}_effective"].append(effective)

            if label == "baseline":
                baseline_output = output
            else:
                candidate_output = output

    records["baseline_output"] = baseline_output
    records["candidate_output"] = candidate_output
    return records


def print_pair(name, paired, comparison):
    baseline = summarize(paired["baseline_wall"])
    candidate = summarize(paired["candidate_wall"])
    baseline_effective = summarize(paired["baseline_effective"])
    candidate_effective = summarize(paired["candidate_effective"])

    change = 100.0 * (
        candidate["median_s"] / baseline["median_s"] - 1.0
    )

    print()
    print(name)
    print(
        "  baseline:",
        f"{baseline['median_s']:.3f} s",
        f"(IQR {baseline['iqr_s']:.3f} s)",
    )
    print(
        "  candidate:",
        f"{candidate['median_s']:.3f} s",
        f"(IQR {candidate['iqr_s']:.3f} s)",
    )
    print("  runtime change:", f"{change:+.2f}%")
    print(
        "  effective cores:",
        f"{baseline_effective['median_s']:.2f}",
        "->",
        f"{candidate_effective['median_s']:.2f}",
    )
    print("  max_abs:", f"{comparison['max_abs']:.3e}")
    print("  rel_l2:", f"{comparison['rel_l2']:.3e}")

    return {
        "baseline": baseline,
        "candidate": candidate,
        "baseline_effective_cores": baseline_effective,
        "candidate_effective_cores": candidate_effective,
        "runtime_change_percent": float(change),
        "comparison": comparison,
    }


## Build the full-year waveform, WDM grid, and reusable one-arm plan

In [ ]:

# --- 4. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}


In [ ]:

# --- 5. Generate and pad the waveform ---

started = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = (
    generate_windowed_padded_waveform(
        waveform_model="emri",
        t0=t0,
        dt=dt,
        N=N,
        model_kwargs=model_kwargs,
        window_function="tukey",
        window_kwargs=window_kwargs,
        wdm_Nf=Nf,
        return_padding_info=True,
    )
)

waveform_generation_s = time.perf_counter() - started

print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
print("times_pad:", times_pad.shape, times_pad.dtype)
print("hplus_td:", hplus_td.shape, hplus_td.dtype)
print("hcross_td:", hcross_td.shape, hcross_td.dtype)
print("WDM Nt x Nf:", pad_info["wdm_Nt"], "x", Nf)
print("RSS:", f"{rss_gib():.2f} GiB")


In [ ]:

# --- 6. Build the WDM grid and reusable one-arm plan ---

started = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - started

started = time.perf_counter()
one_arm_plan = OneArmWDMPlan.build_from_context(
    grid_context,
    orbits_trailing,
    theta,
    phi,
    L_trunc=L_TRUNC,
    variable_shift_kwargs=SHIFT_KWARGS,
    use_jax=True,
)
plan_build_s = time.perf_counter() - started

shift_plan = one_arm_plan.shift_plan

assert shift_plan.resolved_assembly_backend == ASSEMBLY_BACKEND
assert shift_plan.resolved_assembly_precision == ASSEMBLY_PRECISION
assert shift_plan.config.row_chunk_size == ROW_CHUNK_SIZE
assert shift_plan.config.lag_block_size == LAG_BLOCK_SIZE
assert shift_plan.num_jobs == 6

print("WDM grid build_s:", f"{grid_build_s:.3f}")
print("One-arm plan build_s:", f"{plan_build_s:.3f}")
print("WDM shape:", (grid_context["wdm"].Nt, grid_context["wdm"].Nf))
print("Plan jobs:", shift_plan.num_jobs)
print("Persistent plan memory_MiB:", shift_plan.device_plan_memory_bytes / 1024**2)
print("RSS:", f"{rss_gib():.2f} GiB")


## Parallel plus/cross WDM transforms

In [ ]:

# --- 7. Sequential versus parallel DWTs ---

def run_sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )


def run_parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )


dwt_paired = paired_benchmark(
    run_sequential_dwt,
    run_parallel_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)

dwt_plus_comparison = compare_arrays(
    dwt_paired["candidate_output"]["w_t_plus"],
    dwt_paired["baseline_output"]["w_t_plus"],
)
dwt_cross_comparison = compare_arrays(
    dwt_paired["candidate_output"]["w_t_cross"],
    dwt_paired["baseline_output"]["w_t_cross"],
)
dwt_comparison = {
    "max_abs": max(
        dwt_plus_comparison["max_abs"],
        dwt_cross_comparison["max_abs"],
    ),
    "rel_l2": max(
        dwt_plus_comparison["rel_l2"],
        dwt_cross_comparison["rel_l2"],
    ),
}

dwt_result = print_pair(
    "Parallel plus/cross DWT",
    dwt_paired,
    dwt_comparison,
)

assert dwt_comparison["rel_l2"] == 0.0

selected_context = dwt_paired["candidate_output"]
w_plus_device = jnp.asarray(
    selected_context["w_t_plus"],
    dtype=jnp.complex64,
)
w_cross_device = jnp.asarray(
    selected_context["w_t_cross"],
    dtype=jnp.complex64,
)
block_result((w_plus_device, w_cross_device))


## Parallel one-arm link groups

In [ ]:

# --- 8. Ordinary six-link shift versus two groups of three links ---

def run_ordinary_shift():
    return one_arm_plan.apply_device(
        w_plus_device,
        w_cross_device,
        execution_mode="materialized",
    )


def run_parallel_shift():
    return one_arm_plan.apply_device(
        w_plus_device,
        w_cross_device,
        execution_mode="materialized_parallel_links",
        parallel_links_per_group=PARALLEL_LINKS_PER_GROUP,
        parallel_max_workers=PARALLEL_MAX_WORKERS,
    )


shift_paired = paired_benchmark(
    run_ordinary_shift,
    run_parallel_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)

shift_comparison = compare_arrays(
    shift_paired["candidate_output"],
    shift_paired["baseline_output"],
)
shift_result = print_pair(
    "Two groups × three directed links",
    shift_paired,
    shift_comparison,
)

assert shift_comparison["rel_l2"] == 0.0


## Complete waveform-dependent one-arm stage

In [ ]:

# --- 9. Sequential ordinary A1 versus selected parallel A1 ---

def run_ordinary_pipeline():
    context = run_sequential_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        execution_mode="materialized",
    )


def run_parallel_pipeline():
    context = run_parallel_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        execution_mode="materialized_parallel_links",
        parallel_links_per_group=PARALLEL_LINKS_PER_GROUP,
        parallel_max_workers=PARALLEL_MAX_WORKERS,
    )


pipeline_paired = paired_benchmark(
    run_ordinary_pipeline,
    run_parallel_pipeline,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)

pipeline_comparison = compare_arrays(
    pipeline_paired["candidate_output"],
    pipeline_paired["baseline_output"],
)
pipeline_result = print_pair(
    "Complete A1: parallel DWT + parallel links",
    pipeline_paired,
    pipeline_comparison,
)

assert pipeline_comparison["rel_l2"] == 0.0


## Production API profile and saved result

In [ ]:

# --- 10. Verify profiling metadata through the public one-arm API ---

profiled_output, public_profile = one_arm_plan.apply_device(
    w_plus_device,
    w_cross_device,
    execution_mode="materialized_parallel_links",
    parallel_links_per_group=PARALLEL_LINKS_PER_GROUP,
    parallel_max_workers=PARALLEL_MAX_WORKERS,
    return_profile=True,
)
block_result(profiled_output)

public_comparison = compare_arrays(
    profiled_output,
    shift_paired["candidate_output"],
)

print("Public parallel-link profile:")
for key, value in public_profile.items():
    print(f"  {key}: {value}")
print("Public API comparison:", public_comparison)

assert public_comparison["rel_l2"] == 0.0
assert public_profile["parallel_group_size"] == PARALLEL_LINKS_PER_GROUP
assert public_profile["parallel_num_groups"] == 2
assert public_profile["parallel_max_workers"] == PARALLEL_MAX_WORKERS


In [ ]:

# --- 11. Compact summary ---

results = {
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "q_wdm": q_wdm,
        "L_TRUNC": L_TRUNC,
        "ROW_CHUNK_SIZE": ROW_CHUNK_SIZE,
        "LAG_BLOCK_SIZE": LAG_BLOCK_SIZE,
        "ASSEMBLY_PRECISION": ASSEMBLY_PRECISION,
        "PARALLEL_LINKS_PER_GROUP": PARALLEL_LINKS_PER_GROUP,
        "PARALLEL_MAX_WORKERS": PARALLEL_MAX_WORKERS,
    },
    "setup": {
        "waveform_generation_s": waveform_generation_s,
        "grid_build_s": grid_build_s,
        "plan_build_s": plan_build_s,
        "persistent_plan_memory_bytes": (
            shift_plan.device_plan_memory_bytes
        ),
    },
    "dwt": dwt_result,
    "shift": shift_result,
    "complete_a1": pipeline_result,
    "public_profile": public_profile,
}

print()
print("=" * 80)
print("Selected production route")
print("=" * 80)
print("parallel_wdm_transforms=True")
print("one_arm_execution_mode='materialized_parallel_links'")
print(
    "one_arm_parallel_links_per_group="
    f"{PARALLEL_LINKS_PER_GROUP}"
)
print(
    "one_arm_parallel_max_workers="
    f"{PARALLEL_MAX_WORKERS}"
)
print(
    "Complete A1 median:",
    f"{pipeline_result['candidate']['median_s']:.3f} s",
)
print(
    "Complete A1 runtime change:",
    f"{pipeline_result['runtime_change_percent']:+.2f}%",
)
print(
    "Complete A1 rel_l2:",
    f"{pipeline_result['comparison']['rel_l2']:.3e}",
)

if SAVE_RESULTS:
    RESULT_PATH.write_text(
        json.dumps(results, indent=2),
        encoding="utf-8",
    )
    print("Saved:", RESULT_PATH)
